## Governança e Unity Catalog

Este notebook tem como objetivo demonstrar os recursos de governança utilizados no projeto NYC Taxi por meio do Unity Catalog.

Serão analisados a organização dos dados em catálogos e schemas, o gerenciamento dos objetos de dados, as permissões de acesso e o rastreamento de linhagem (lineage) entre as diferentes camadas do pipeline.

A análise busca demonstrar como o Unity Catalog permite centralizar a governança dos dados utilizados no pipeline Batch e Streaming.


**1 - Configuração e validação do ambiente**

Inicialmente será validado o catálogo utilizado pelo projeto e sua estrutura de schemas no Unity Catalog.

**Catálogo Atual**

In [0]:
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+



**Catálogos Disponíveis**

In [0]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|aulas_bigdata|
|nyc_taxi_data|
|      samples|
|       system|
|    workspace|
+-------------+



**Schemas do projeto**

In [0]:
spark.sql("SHOW SCHEMAS IN nyc_taxi_data").show()

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|               raw|
|            silver|
+------------------+



**Estrutura do ambiente**

O projeto utiliza o catálogo `nyc_taxi_data` como estrutura central de governança dos dados.

Dentro do catálogo, os dados do pipeline estão organizados principalmente nos seguintes schemas:

- `raw`: armazenamento dos arquivos utilizados no processo de ingestão;
- `bronze`: dados ingeridos e persistidos sem transformações;
- `silver`: dados tratados, validados e enriquecidos;
- `gold`: dados agregados e preparados para consumo analítico.

Essa organização permite separar logicamente as diferentes etapas do pipeline e centralizar a governança dos objetos por meio do Unity Catalog.

**2 - Estrutura dos objetos no Unity Catalog**

Após a validação do catálogo e dos schemas, será analisada a organização dos objetos utilizados pelo pipeline.

O Unity Catalog utiliza uma estrutura hierárquica composta por catálogo, schema e objeto. No projeto, o catálogo `nyc_taxi_data` centraliza os objetos das diferentes camadas da arquitetura de dados.

**Camada Bronze**

A camada Bronze contém os dados ingeridos pelo pipeline antes das principais transformações e enriquecimentos.

In [0]:
spark.sql("SHOW TABLES IN nyc_taxi_data.bronze").show(truncate=False)

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|bronze  |viagens          |false      |
|bronze  |viagens_streaming|false      |
|bronze  |zonas            |false      |
+--------+-----------------+-----------+



**Camada Silver**

A camada Silver contém os dados tratados, validados e enriquecidos durante o processamento do pipeline.

In [0]:
spark.sql("SHOW TABLES IN nyc_taxi_data.silver").show(truncate=False)

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|silver  |viagens  |false      |
+--------+---------+-----------+



**Camada Gold**

A camada Gold contém tabelas derivadas da camada Silver e preparadas para consumo analítico, incluindo agregações, rankings, pivotação e indicadores de negócio.

In [0]:
spark.sql("SHOW TABLES IN nyc_taxi_data.gold").show(truncate=False)

+--------+---------------------------+-----------+
|database|tableName                  |isTemporary|
+--------+---------------------------+-----------+
|gold    |ranking_mensal_distritos   |false      |
|gold    |viagens_categoria_distancia|false      |
|gold    |viagens_mensal             |false      |
|gold    |viagens_pivot_distrito_mes |false      |
|gold    |viagens_streaming_watermark|false      |
|gold    |viagens_zona_destino       |false      |
|gold    |viagens_zona_origem        |false      |
+--------+---------------------------+-----------+



**Camada Raw**

A camada Raw é utilizada para armazenamento dos arquivos utilizados nos processos de ingestão. Diferentemente das demais camadas, essa área utiliza um Volume do Unity Catalog para gerenciamento dos arquivos.

In [0]:
spark.sql("SHOW VOLUMES IN nyc_taxi_data.raw").show(truncate=False)

+--------+-----------+
|database|volume_name|
+--------+-----------+
|raw     |landing    |
+--------+-----------+



**Organização das camadas**

A estrutura do catálogo demonstra a separação lógica das diferentes etapas do pipeline:

`raw → bronze → silver → gold`

A camada `raw` utiliza um Volume para armazenamento dos arquivos de ingestão, enquanto as camadas `bronze`, `silver` e `gold` utilizam tabelas para representar os diferentes estágios de processamento.

Essa organização permite centralizar os objetos do projeto no catálogo `nyc_taxi_data`, mantendo separadas as responsabilidades de ingestão, tratamento, enriquecimento e disponibilização analítica dos dados.

**3 - Gerenciamento e armazenamento dos dados**

Nesta seção será analisada a forma como os objetos do projeto são armazenados e gerenciados pelo Unity Catalog.

A análise permitirá identificar a diferença entre tabelas gerenciadas (Managed Tables), cujo ciclo de vida e localização são administrados pelo Unity Catalog, e objetos externos, cuja localização de armazenamento é definida externamente.

**Análise da tabela Silver**

Inicialmente será analisada a tabela `nyc_taxi_data.silver.viagens` para identificar seu formato, tipo e características de armazenamento.

In [0]:
%sql
DESCRIBE EXTENDED nyc_taxi_data.silver.viagens;

col_name,data_type,comment
Id_Fornecedor_Tecnologia,int,null
Inicio_Corrida,timestamp_ntz,null
Fim_Corrida,timestamp_ntz,null
Qtd_Passageiros,bigint,null
Distancia_corrida_milhas,double,null
Id_tarifa,bigint,null
Flag_armazenamento,string,null
Zona_inicio_corrida,int,null
Zona_fim_corrida,int,null
Id_tipo_pagamento,bigint,null


**Características físicas da tabela**

O comando `DESCRIBE DETAIL` será utilizado para consultar metadados da tabela, como formato de armazenamento, localização, quantidade de arquivos e propriedades do Delta Lake.

In [0]:
%sql
DESCRIBE DETAIL nyc_taxi_data.silver.viagens;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,67486b34-a50b-470a-8fd8-a5e158e658ec,nyc_taxi_data.silver.viagens,null,,2026-08-15T22:47:55.925Z,2026-08-25T17:58:50.000Z,List(),List(),3,226972148,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants, timestampNtz)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


**Verificando o Volume Raw**

In [0]:
%sql
DESCRIBE VOLUME nyc_taxi_data.raw.landing;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
landing,nyc_taxi_data,raw,leonardo.froes@al.infnet.edu.br,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


**External Locations**

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment


**Análise do gerenciamento dos dados**

A análise da tabela `nyc_taxi_data.silver.viagens` demonstrou que ela possui as seguintes características:

- **Type = MANAGED:** a tabela é gerenciada pelo Unity Catalog;
- **Provider = delta:** os dados utilizam o formato Delta Lake;
- **Is_managed_location = true:** o armazenamento da tabela está associado a uma localização gerenciada pelo ambiente;
- **Statistics:** aproximadamente 227 MB e 11.198.026 registros.

Como a tabela é do tipo `MANAGED`, o gerenciamento do armazenamento e do ciclo de vida dos dados é realizado pelo Unity Catalog.

Também foi executado o comando `SHOW EXTERNAL LOCATIONS`, não sendo identificadas External Locations configuradas no ambiente utilizado pelo projeto.

Dessa forma, o projeto utiliza tabelas gerenciadas pelo Unity Catalog. A utilização de dados externos foi analisada conceitualmente, porém não foi implementada por meio de uma External Location neste ambiente.

**Evidenciando o tipo, "provider" e "is_managed" de tabelas das camadas bronze e gold**

In [0]:
from pyspark.sql.functions import *

tabelas_validacao = [
    "nyc_taxi_data.bronze.viagens",
    "nyc_taxi_data.silver.viagens",
    "nyc_taxi_data.gold.viagens_mensal"
]

for tabela in tabelas_validacao:
    print(f"\nTabela: {tabela}")

    (
        spark.sql(f"DESCRIBE EXTENDED {tabela}")
        .filter(col("col_name").isin("Type", "Provider", "Is_managed_location"))
        .show(truncate=False)
    )


Tabela: nyc_taxi_data.bronze.viagens
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|Type               |MANAGED  |       |
|Provider           |delta    |       |
|Is_managed_location|true     |       |
+-------------------+---------+-------+


Tabela: nyc_taxi_data.silver.viagens
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|Type               |MANAGED  |       |
|Provider           |delta    |       |
|Is_managed_location|true     |       |
+-------------------+---------+-------+


Tabela: nyc_taxi_data.gold.viagens_mensal
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|Type               |MANAGED  |       |
|Provider           |delta    |       |
|Is_managed_location|true     |       |
+-------------------+---------+-------+



**4 - Permissões e controle de acesso**

O Unity Catalog permite centralizar o controle de acesso aos objetos de dados por meio de privilégios atribuídos em diferentes níveis da hierarquia, como catálogo, schema e tabela.

Nesta seção serão analisadas as permissões existentes nos objetos utilizados pelo projeto, buscando demonstrar como o acesso aos dados pode ser controlado de forma centralizada.

**Verificando as permissões do Catálogo como um todo**

In [0]:
%sql
SHOW GRANTS ON CATALOG nyc_taxi_data;

Principal,ActionType,ObjectType,ObjectKey
account users,BROWSE,CATALOG,nyc_taxi_data


**Permissões dos schemas**

Além das permissões definidas no catálogo, o Unity Catalog permite aplicar privilégios individualmente aos schemas, possibilitando diferentes níveis de acesso às camadas do pipeline.

In [0]:
%sql
SHOW GRANTS ON SCHEMA nyc_taxi_data.bronze;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
SHOW GRANTS ON SCHEMA nyc_taxi_data.silver;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
SHOW GRANTS ON SCHEMA nyc_taxi_data.gold;

Principal,ActionType,ObjectType,ObjectKey


**Permissões no nível de tabela**

O controle de acesso também pode ser aplicado diretamente sobre tabelas específicas. Para demonstrar esse nível de granularidade, serão analisadas as permissões da principal tabela da camada Silver.

In [0]:
%sql
SHOW GRANTS ON TABLE nyc_taxi_data.silver.viagens;

Principal,ActionType,ObjectType,ObjectKey


**Análise das permissões**

A consulta de permissões no catálogo `nyc_taxi_data` identificou o privilégio `BROWSE` atribuído ao principal `account users`.

Esse privilégio permite a descoberta dos objetos do catálogo, sem representar necessariamente permissão de leitura ou modificação dos dados.

Ao consultar diretamente as permissões da tabela `nyc_taxi_data.silver.viagens`, não foram encontrados grants explícitos no nível da tabela.

O resultado demonstra que o Unity Catalog permite controlar permissões em diferentes níveis da hierarquia de objetos, incluindo catálogo, schema e tabela, além de possibilitar a utilização de ownership e privilégios herdados.

In [0]:
%sql
SHOW GRANTS ON SCHEMA nyc_taxi_data.silver;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
SHOW GRANTS ON SCHEMA nyc_taxi_data.gold;

Principal,ActionType,ObjectType,ObjectKey


### Conclusão sobre permissões

A análise das permissões mostrou que o catálogo `nyc_taxi_data` possui o privilégio `BROWSE` atribuído ao principal `account users`.

Nos schemas `silver` e `gold`, assim como na tabela `nyc_taxi_data.silver.viagens`, não foram identificados grants explícitos.

Esse resultado demonstra que o Unity Catalog permite administrar privilégios em diferentes níveis da hierarquia, embora neste ambiente de projeto não tenham sido configuradas permissões específicas adicionais por schema ou tabela.

**5 - Linhagem de dados**

O Unity Catalog permite rastrear a linhagem dos dados, identificando as relações entre os objetos utilizados ao longo do pipeline.

Nesta seção será analisada a linhagem das tabelas do projeto, buscando demonstrar o fluxo dos dados entre as camadas Bronze, Silver e Gold e identificar as dependências existentes entre os diferentes objetos.

**Análise da linhagem do pipeline**

A funcionalidade de Lineage do Unity Catalog foi utilizada para visualizar as dependências existentes entre as tabelas das diferentes camadas do pipeline.

A análise da tabela `nyc_taxi_data.silver.viagens` demonstrou que seus dados possuem como origem as tabelas `nyc_taxi_data.bronze.viagens` e `nyc_taxi_data.bronze.zonas`.

A tabela Silver, por sua vez, é utilizada como fonte para diferentes tabelas analíticas da camada Gold, permitindo rastrear a evolução dos dados ao longo do pipeline.

![image_1787684293350.png](./image_1787684293350.png "image_1787684293350.png")

###Resultado da análise

A linhagem registrada pelo Unity Catalog permite visualizar o fluxo de dados entre as camadas:

`Bronze → Silver → Gold`

Na camada Bronze encontram-se os dados de viagens e de zonas utilizados como fontes do processamento.

Na camada Silver, os dados de viagens são tratados, classificados quanto à qualidade e enriquecidos com informações geográficas provenientes da tabela de zonas.

A partir da tabela Silver são produzidas diferentes tabelas Gold destinadas ao consumo analítico, incluindo agregações mensais, análises geográficas, rankings, pivotação e classificação das viagens.

O grafo de linhagem demonstra que o Unity Catalog registra as dependências entre os objetos e possibilita rastrear a origem e a evolução dos dados ao longo do pipeline.

## 6 - Conclusão da Governança

A implementação do projeto utilizou o Unity Catalog como camada central de governança dos dados, permitindo organizar e rastrear os objetos utilizados ao longo do pipeline.

O catálogo `nyc_taxi_data` foi estruturado em schemas correspondentes às diferentes etapas do processamento:

- `raw`: armazenamento dos arquivos utilizados nos processos de ingestão;
- `bronze`: persistência dos dados ingeridos em Batch e Streaming;
- `silver`: dados tratados, validados e enriquecidos;
- `gold`: dados agregados e preparados para consumo analítico.

As tabelas analisadas são gerenciadas pelo Unity Catalog e utilizam Delta Lake como formato de armazenamento. A tabela `nyc_taxi_data.silver.viagens`, por exemplo, foi identificada como `MANAGED`, com `Provider = delta` e `Is_managed_location = true`.

Também foi analisado o modelo de controle de acesso do Unity Catalog. O catálogo possui o privilégio `BROWSE` atribuído ao principal `account users`, enquanto não foram identificados grants explícitos adicionais nos schemas Silver e Gold ou na tabela Silver analisada.

Não foram identificadas External Locations configuradas no ambiente utilizado pelo projeto. Dessa forma, a solução implementada utiliza objetos e localizações gerenciados pelo Unity Catalog, enquanto a distinção entre armazenamento gerenciado e externo foi analisada sem a implementação de uma External Location.

Por fim, a funcionalidade de Lineage permitiu visualizar as dependências entre as diferentes camadas do pipeline. Foi possível rastrear as tabelas `bronze.viagens` e `bronze.zonas` como fontes da tabela `silver.viagens` e, posteriormente, a utilização da camada Silver como origem das diferentes tabelas analíticas da camada Gold.

Dessa forma, a solução demonstra a utilização do Unity Catalog para organização dos dados, gerenciamento dos objetos, controle de acesso e rastreamento da linhagem de dados entre as camadas Bronze, Silver e Gold.